# Sliding window


In [ ]:
import numpy as np
from PIL import Image
def sliding_window(image: np.ndarray, window_size: tuple, stride: int):
    win_h, win_w = window_size
    img_h, img_w = image.shape[:2]

    for y in range(0, img_h - win_h + 1, stride):
        for x in range(0, img_w - win_w + 1, stride):
            crop = image[y:y+win_h, x:x+win_w]
            yield x, y, crop

def is_coin(crop, threshold_varians=300):
    varians = np.var(crop)
    if varians > threshold_varians:
        return True, varians
    return False, 0


def compute_iou(box_a, box_b):
    inter_x1 = max(box_a[0], box_b[0])
    inter_y1 = max(box_a[1], box_b[1])
    inter_x2 = min(box_a[2], box_b[2])
    inter_y2 = min(box_a[3], box_b[3])

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    if inter_area == 0:
        return 0.0

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union_area = area_a + area_b - inter_area

    return inter_area / union_area

def non_maximum_suppression(boxes, scores, iou_threshold=0.3):
    if len(boxes) == 0:
        return []

    boxes = np.array(boxes)
    scores = np.array(scores)

    picked_boxes = []

    order = np.argsort(scores)[::-1]

    while len(order) > 0:
        current_idx = order[0]
        picked_boxes.append(boxes[current_idx].tolist())

        boxes_to_keep = []
        for i in range(1, len(order)):
            idx_compare = order[i]
            iou = compute_iou(boxes[current_idx], boxes[idx_compare])

            if iou < iou_threshold:
                boxes_to_keep.append(idx_compare)

        order = boxes_to_keep

    return picked_boxes

def detection_eval(predictions, ground_truths, iou_threshold=0.5):
    TP, FP = 0, 0
    matched_gt = set()
    iou_scores = []

    for pred_box in predictions:
        best_iou = 0.0
        best_gt_idx = -1

        for i, gt_box in enumerate(ground_truths):
            if i in matched_gt:
                continue
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = i

        if best_iou >= iou_threshold:
            TP += 1
            matched_gt.add(best_gt_idx)
            iou_scores.append(best_iou)
        else:
            FP += 1

    FN = len(ground_truths) - len(matched_gt)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    mean_iou = sum(iou_scores) / len(iou_scores) if iou_scores else 0.0

    return {"TP": TP, "FP": FP, "FN": FN, "Precision": precision, "Recall": recall, "Mean_IoU": mean_iou}

if __name__ == "__main__":
    try:
        image = Image.open("coin.jpeg").convert('L')
        img_np = np.array(image)
        print(f"Gambar dimuat dengan resolusi: {img_np.shape}")
    except FileNotFoundError:
        print("GAGAL: File gambar tidak ditemukan. Pastikan nama file benar.")
        exit()

    ground_truths = [

    [195, 5, 225, 25],
    [45, 25, 75, 45],
    [85, 25, 120, 45],
    [145, 15, 180, 35],
    [245, 25, 285, 45],

    [0, 85, 40, 105],
    [55, 80, 105, 110],
    [125, 80, 170, 105],
    [180, 50, 230, 80],
    [260, 65, 295, 90],

    [55, 130, 105, 160],
    [140, 125, 175, 145],
    [210, 110, 255, 135]

    ]

    window_size = (80, 80)
    stride = 20

    raw_predictions = []
    raw_scores = []

    for x, y, crop in sliding_window(img_np, window_size, stride):
        is_obj, score = is_coin(crop, threshold_varians=500)

        if is_obj:
            box = [x, y, x + window_size[1], y + window_size[0]]
            raw_predictions.append(box)
            raw_scores.append(score)

    print(f"Prediksi mentah didapat: {len(raw_predictions)} kotak.")

    final_predictions = non_maximum_suppression(raw_predictions, raw_scores, iou_threshold=0.3)
    print(f"Prediksi final setelah NMS: {len(final_predictions)} kotak.")

    hasil = detection_eval(final_predictions, ground_truths, iou_threshold=0.2)

    print("\n--- LAPORAN ANALISIS AKHIR ---")
    for k, v in hasil.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

Gambar dimuat dengan resolusi: (168, 299)
Prediksi mentah didapat: 48 kotak.
Prediksi final setelah NMS: 9 kotak.

--- LAPORAN ANALISIS AKHIR ---
TP: 2
FP: 7
FN: 11
Precision: 0.2222
Recall: 0.1538
Mean_IoU: 0.2202
